# K-Means on Processed EEG Data

This notebook trains and evaluates the TensorFlow `KMeansTF` model on your **processed / shuffled** EEG dataset.

- **Input**: The folder you selected via the GUI file shuffler (all `.txt` files there are now `.csv`).
- **Goal**: Reuse the best K-Means pipeline we built earlier (per-file windows, bandpass filtering, brainwave bands + temporal features, Z-score scaling, optional PCA) but pointed at the **processed data root**.

> **Note:** Update `data_root_processed` in the next cell to match the folder that contains your processed EEG files (e.g. `"/home/elizabeth/Documents/Data_clean/Data_clean"` or your `Data_clean_processed` path).

In [1]:
# Imports (robust path setup so kmeans_model is found)
print("Starting Imports...")

import sys
import os
import re
import pathlib
import importlib
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import skew, kurtosis

# Ensure the directory containing kmeans_model.py is on sys.path
_nb_dir = pathlib.Path.cwd()
_candidates = [
    _nb_dir,                                      # e.g. repo root if kmeans_model.py is here
    _nb_dir / "prediction_k_means" / "tensorflow",  # repo-root-based path
    _nb_dir / "tensorflow",                      # if cwd is already prediction_k_means
]

_kmeans_dir = None
for cand in _candidates:
    if (cand / "kmeans_model.py").exists():
        _kmeans_dir = cand
        break

if _kmeans_dir is not None and str(_kmeans_dir) not in sys.path:
    sys.path.insert(0, str(_kmeans_dir))
else:
    print("Warning: could not automatically locate kmeans_model.py; check paths if import fails.")

print("Loading kmeans_model...")

import kmeans_model
kmeans_model = importlib.reload(kmeans_model)

print("Imports OK")

Starting Imports...
Loading kmeans_model...


2026-03-09 15:53:03.255439: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-09 15:53:03.257058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 15:53:03.582016: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-09 15:53:12.862579: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

Imports OK


In [2]:
# Configuration

# Set this to the folder that now holds your **processed/shuffled** EEG files.
# This folder should now contain 6 subfolders (backward, forward, landing, left, right, takeoff)
# and a single report file named "remove-8channel-Report.csv" which will be ignored.
data_root_processed = pathlib.Path("/home/elizabeth/Documents/Avatar-Elizabeth-Forked/Data_clean/Data_clean/processed")

# Default sampling rate for processed files (can be changed if you ever
# export at a different Hz in the future).
default_fs = 125.0

# Directories to skip entirely (leave empty unless you have specific folders to ignore)
skip_dirs = set()

# Filenames to ignore even if they live under data_root_processed
ignore_files = {"remove-8channel-Report.csv"}

# K-Means / PCA settings
use_pca = False
pca_components = 50
n_clusters = 150
max_iter = 1000
tol = 1e-6
random_state = 42

print("Using processed data root:", data_root_processed)
print("Configuration Done!")

Using processed data root: /home/elizabeth/Documents/Avatar-Elizabeth-Forked/Data_clean/Data_clean/processed
Configuration Done!


In [3]:
# Load all processed EEG files and attach a per-file sampling rate.

core_dir = pathlib.Path(data_root_processed)
assert core_dir.exists(), f"Processed data root does not exist: {core_dir}"

dfs = []
fs_values = []

# We know the processed root contains exactly these 6 label folders, so
# we can iterate them directly instead of doing a full recursive walk.
expected_labels = ["backward", "forward", "landing", "left", "right", "takeoff"]

for label_dir in core_dir.iterdir():
    if not label_dir.is_dir():
        continue
    if label_dir.name not in expected_labels:
        continue

    for item in label_dir.glob("*.csv"):
        # Skip known non‑data report file(s)
        if item.name in ignore_files:
            continue

        try:
            # Processed CSV files from the GUI pipeline currently use a fixed sampling rate
            # but we leave this configurable via `default_fs` in case you change it later.
            file_fs = float(default_fs)

            # CSV produced by the shuffler: already stripped of comment lines, header row is on line 1
            df = pd.read_csv(item, sep=",", on_bad_lines="skip")

            if df.empty or df.shape[1] < 2:
                continue

            df["src_filename"] = str(item)
            df["fs"] = file_fs
            dfs.append(df)
            fs_values.append(file_fs)
        except Exception as e:
            print(f"Failed to read {item}: {e}")

print(f"Read {len(dfs)} files from processed data root")

eeg_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print("Combined shape:", eeg_data.shape)

if fs_values:
    fs_series = pd.Series(fs_values)
    print("Per-file sample rates (Hz):")
    print(fs_series.value_counts().sort_index())

print("All EEG Files Loaded!")

Read 1254 files from processed data root
Combined shape: (1606736, 35)
Per-file sample rates (Hz):
125.0    1254
Name: count, dtype: int64
All EEG Files Loaded!


In [4]:
# Derive labels directly from folder names under the processed root.

assert not eeg_data.empty, "No data loaded. Check data_root_processed points at your processed folder."

expected_labels = {"backward", "forward", "landing", "left", "right", "takeoff"}

src_paths = eeg_data["src_filename"].astype(str)
labels = []
for p in src_paths:
    parts = pathlib.Path(p).parts
    # Find the first part that matches one of the expected label folder names
    lbl = ""
    for part in parts:
        pl = part.lower()
        if pl in expected_labels:
            lbl = pl
            break
    labels.append(lbl)

eeg_data["label_txt"] = pd.Series(labels, index=eeg_data.index).astype(str)

print(eeg_data.groupby(["label_txt"])["src_filename"].count())
print("Labeling Complete!")

label_txt
backward    279619
forward     272609
landing     299873
left        297775
right       297856
takeoff     159004
Name: src_filename, dtype: int64
Labeling Complete!


In [5]:
# Optional: inspect per-channel EEG noise to help decide which channels to drop.

# We compute simple statistics per EXG channel across the entire processed dataset.
# You can use this table to decide which channels (if any) to treat as "noisy".

numeric_cols_all = eeg_data.select_dtypes(include=[np.number]).columns
# Heuristic: EEG channels are those whose column name contains "EXG".
eeg_cols_all = [c for c in numeric_cols_all if "EXG" in c]

channel_stats = pd.DataFrame(index=eeg_cols_all)
channel_stats["std"] = eeg_data[eeg_cols_all].std()
channel_stats["median_abs"] = eeg_data[eeg_cols_all].median().abs()
channel_stats["max_abs"] = eeg_data[eeg_cols_all].abs().max()

channel_stats = channel_stats.sort_values("std", ascending=False)

print("Top EEG channels by standard deviation (potentially noisy):")
display(channel_stats.head(20))

# Automatically identify channels hitting the OpenBCI saturation rail (187,500 uV)
# plus the ones we manually identified as extreme.
railed_channels = channel_stats[channel_stats["max_abs"] >= 187500].index.tolist()

# Combine manual drops with auto-detected railed channels
noisy_channels = list(set(railed_channels))

print(f"Channels identified for dropping: {noisy_channels}")


Top EEG channels by standard deviation (potentially noisy):


,std,median_abs,max_abs
EXG Channel 11,499744.489300,10313.519575,2.448264e+06
EXG Channel 15,82045.649955,25919.976940,1.875000e+05
EXG Channel 2,63455.443731,24692.352079,1.875000e+05
EXG Channel 14,55536.999002,17533.043627,1.875000e+05
EXG Channel 3,48169.076361,12462.148066,1.875000e+05
EXG Channel 5,45588.819036,19.781294,1.875000e+05
EXG Channel 13,40641.456301,15027.658943,1.875000e+05
EXG Channel 10,31918.038876,19851.299507,1.875000e+05
EXG Channel 0,29170.751701,11995.913028,1.777953e+05
EXG Channel 7,28461.178831,2971.843239,1.508448e+05


Channels identified for dropping: [' EXG Channel 2', ' EXG Channel 5', ' EXG Channel 13', ' EXG Channel 11', ' EXG Channel 3', ' EXG Channel 14', ' EXG Channel 10', ' EXG Channel 15']


In [6]:
# Build feature matrix X and label vector y from processed data

df = eeg_data.copy()
# Keep all sampling rates; `fs` is used later on a per-file basis.
df = df[df["label_txt"].astype(str).str.len() > 0]

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Drop obvious non-EEG columns but keep accelerometer; do not use sampling rate as a feature
drop_keywords = ["Sample", "Timestamp", "Marker", "Not Used", "Analog", "Digital", "Other"]
drop_cols = [c for c in num_cols if any(key in c for key in drop_keywords)]

if "fs" in num_cols:
    drop_cols.append("fs")

# src_filename is non-numeric, but guard just in case
if "src_filename" in num_cols:
    drop_cols.append("src_filename")

drop_cols = sorted(set(drop_cols))
feature_cols = [c for c in num_cols if c not in drop_cols]

feat = df[feature_cols].copy()
feat = feat.replace([np.inf, -np.inf], np.nan)
feat = feat.dropna(axis=1, how="all")

med = feat.median(numeric_only=True)
feat = feat.fillna(med)

std = feat.std(numeric_only=True)
keep = std[std > 0].index.tolist()
if len(keep) == 0:
    keep = feature_cols
    print("Warning: no columns had std > 0; using all numeric columns.")

# Drop any channels explicitly marked as noisy in the previous cell
if "noisy_channels" in globals():
    for ch in noisy_channels:
        if ch in keep:
            keep.remove(ch)
            print(f"Dropped noisy channel: {ch}")

# # Drop accelerometer channels (EEG only for K-Means)
# keep = [c for c in keep if "Accel" not in c]

feat = feat[keep]
X = feat.to_numpy(dtype=np.float32)

y_cat = df["label_txt"].astype("category")
y = y_cat.cat.codes.to_numpy(dtype=np.int32)
label_names = list(y_cat.cat.categories)

# Store numeric labels for use in per-file windowing
df["y_code"] = y

print("X shape:", X.shape, "classes:", len(label_names))
print(f"Features being used for K-Means: {keep}")
print("Feature Extraction Complete!")

Dropped noisy channel:  EXG Channel 2
Dropped noisy channel:  EXG Channel 5
Dropped noisy channel:  EXG Channel 13
Dropped noisy channel:  EXG Channel 11
Dropped noisy channel:  EXG Channel 3
Dropped noisy channel:  EXG Channel 14
Dropped noisy channel:  EXG Channel 10
Dropped noisy channel:  EXG Channel 15
X shape: (1606736, 11) classes: 6
Features being used for K-Means: [' EXG Channel 0', ' EXG Channel 1', ' EXG Channel 4', ' EXG Channel 6', ' EXG Channel 7', ' EXG Channel 8', ' EXG Channel 9', ' EXG Channel 12', ' Accel Channel 0', ' Accel Channel 1', ' Accel Channel 2']
Feature Extraction Complete!


In [ ]:
# Feature extraction: per-file 1-second windows, bandpass 1–50 Hz, brainwave bands + temporal texture


X_psd = []
y_psd = []

# Split EEG vs accelerometer using the chosen keep list
eeg_idx = [i for i, c in enumerate(keep) if "Accel" not in c]
accel_idx = [i for i, c in enumerate(keep) if "Accel" in c]

print(f"Extracting features from {len(eeg_idx)} EEG channels and {len(accel_idx)} Accel channels...")

required_cols = keep + ["y_code", "fs", "src_filename"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for PSD extraction: {missing_cols}")

work_df = df[required_cols].copy()

for fname, g in work_df.groupby("src_filename", sort=False):
    file_fs = float(g["fs"].iloc[0]) if "fs" in g.columns else 125.0
    if file_fs <= 0:
        file_fs = 125.0

    # 1. Separate and Clean the entire file first
    # This prevents edge artifacts from filtering small windows
    raw_data = g[keep].to_numpy(dtype=np.float32)
    y_labels = g["y_code"].to_numpy(dtype=np.int32)
    
    # Filter only the EEG portion
    eeg_filtered = kmeans_model.apply_bandpass_to_signal(raw_data[:, eeg_idx], file_fs)
    accel_raw = raw_data[:, accel_idx]
    
    # 2. Windowing
    win_size = int(file_fs)
    step = max(1, win_size // 2)
    
    if eeg_filtered.shape[0] < win_size:
        continue

    for start in range(0, eeg_filtered.shape[0] - win_size, step):
        # Slice windows
        win_eeg = eeg_filtered[start : start + win_size, :]
        win_accel = accel_raw[start : start + win_size, :]

        # Extract features using the .py helper
        row_feats = kmeans_model.extract_window_features(win_eeg, win_accel, file_fs)
        X_psd.append(row_feats)
        y_psd.append(y_labels[start])


X_psd = np.array(X_psd, dtype=np.float32)
y_psd = np.array(y_psd, dtype=np.int32)

print(f"Built {X_psd.shape[0]} windows with {X_psd.shape[1]} features each.")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_psd, y_psd, test_size=0.2, random_state=42, stratify=y_psd
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train/test split complete!")

Extracting features from 8 EEG channels and 3 Accel channels...
Built 23974 windows with 35 features each.
Train shape: (19179, 35) Test shape: (4795, 35)
Train/test split complete!


In [8]:
# Rescale features (Z‑score) and optionally apply PCA

# For relative power features we **skip log10**; they are already normalized.
X_train_proc = X_train
X_test_proc = X_test

mu = np.nanmean(X_train_proc, axis=0)
sd = np.nanstd(X_train_proc, axis=0)

# Avoid divide‑by‑zero and NaNs
sd[sd == 0] = 1.0
mu[np.isnan(mu)] = 0.0

X_train_z = (X_train_proc - mu) / sd
X_test_z = (X_test_proc - mu) / sd

X_train_z = np.nan_to_num(X_train_z)
X_test_z = np.nan_to_num(X_test_z)

print(f"Rescaling complete. Any NaNs left? {np.isnan(X_train_z).any()}")
print("Training shape after scaling:", X_train_z.shape)

# PCA (optional)
if use_pca:
    pca = PCA(n_components=pca_components, random_state=42)
    X_train_final = pca.fit_transform(X_train_z)
    X_test_final = pca.transform(X_test_z)
    print(
        f"PCA complete. Retained {np.sum(pca.explained_variance_ratio_) * 100:.2f}% of variance; "
        f"feature dim = {X_train_final.shape[1]}"
    )
else:
    pca = None
    X_train_final = X_train_z
    X_test_final = X_test_z
    print("PCA disabled; using raw scaled features.")

Rescaling complete. Any NaNs left? False
Training shape after scaling: (19179, 35)
PCA disabled; using raw scaled features.


In [9]:
# Train KMeansTF on processed features and evaluate

model = kmeans_model.KMeansTF(
    n_clusters=n_clusters,
    max_iter=max_iter,
    tol=tol,
    random_state=random_state,
)

print(f"Starting KMeansTF training on {X_train_final.shape[1]} features...")
model.fit(X_train_final, y_train)

print(f"Inertia (tightness): {model.inertia_:.2f}")
print("Cluster-to-label mapping (processed data):")
for k, v in sorted(model.cluster_to_label_map_.items()):
    print(f"  cluster {k} -> {label_names[v]} (class {v})")

print("Predicting on test set...")
y_pred = model.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy (processed data): {acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

Starting KMeansTF training on 35 features...


2026-03-09 16:18:13.865656: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Inertia (tightness): 99962.00
Cluster-to-label mapping (processed data):
  cluster 0 -> backward (class 0)
  cluster 1 -> left (class 3)
  cluster 2 -> right (class 4)
  cluster 3 -> landing (class 2)
  cluster 4 -> backward (class 0)
  cluster 5 -> left (class 3)
  cluster 6 -> forward (class 1)
  cluster 7 -> forward (class 1)
  cluster 8 -> backward (class 0)
  cluster 9 -> backward (class 0)
  cluster 10 -> landing (class 2)
  cluster 11 -> left (class 3)
  cluster 12 -> right (class 4)
  cluster 13 -> right (class 4)
  cluster 14 -> landing (class 2)
  cluster 15 -> landing (class 2)
  cluster 16 -> forward (class 1)
  cluster 17 -> landing (class 2)
  cluster 18 -> left (class 3)
  cluster 19 -> right (class 4)
  cluster 20 -> right (class 4)
  cluster 21 -> left (class 3)
  cluster 22 -> left (class 3)
  cluster 23 -> right (class 4)
  cluster 24 -> forward (class 1)
  cluster 25 -> landing (class 2)
  cluster 26 -> landing (class 2)
  cluster 27 -> backward (class 0)
  cluster 

In [10]:
# Save the trained KMeansTF model + preprocessing metadata for processed data

meta = {
    "mu": mu.astype(np.float32),
    "sd": sd.astype(np.float32),
    "label_names": label_names,
    "n_eeg": len([c for c in keep if "EXG" in c]),
    "n_accel": len([c for c in keep if "Accel" in c]),
    "n_clusters": n_clusters,
    "use_pca": use_pca,
    "pca": pca,
}

out_path = "kmeans_processed_CLEAN.pth"
kmeans_model.save_model(out_path, model, meta)

print(f"Model for PROCESSED data saved to {out_path}")

Model for PROCESSED data saved to kmeans_processed_CLEAN.pth
